# COMP5339 Assignment 1 — Electricity Sector Data Integration

**Team:** [Name A] · [Name B] · [Name C]

Orchestrator notebook. Each section runs one stage of the pipeline; implementation lives under `pipeline/`.

## How to run

1. Run the **Setup** cell to install dependencies.
2. Toggle `USE_CACHED_*` flags in **Configuration** (`True` = use shipped cache, `False` = refetch).
3. **Kernel → Restart & Run All**.

Outputs land under `data/`; final database is `data/curated/warehouse.duckdb`.

In [ ]:
# %%capture
# Install project dependencies.
# Output is suppressed; remove the %%capture line to see pip output.
# %pip install -r requirements.txt

## Configuration

Set a flag to `False` to refetch from source. Leave `USE_CACHED_GEO = True` after the first run — Nominatim is rate-limited and a cold geocode takes ~3 hours.

In [1]:
# ── Pipeline configuration ─────────────────────────────────────
USE_CACHED_NGER = True   # NGER electricity sector data 
USE_CACHED_CER  = True   # CER large-scale renewables (csv download)
USE_CACHED_ABS  = True   # ABS Data by Regions (XLSX download)
USE_CACHED_GEO  = True   # Geocoded facility coordinates (Nominatim)
# ───────────────────────────────────────────────────────────────

In [2]:
from pathlib import Path

import pandas as pd

# Make the local `pipeline/` package importable regardless of the
# notebook's launch directory.
import sys
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ensure data directories exist up-front.
for sub in ("raw", "cleaned", "curated"):
    (PROJECT_ROOT / "data" / sub).mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

---

## 1. Data Acquisition

Modules under `pipeline/acquire/` fetch each source and save raw payloads to `data/raw/<source>/`.

### 1.1 NGER — Annual emissions and generation (CER public data API, FY2014-15 → FY2023-24)

In [3]:
from pipeline.acquire.nger import fetch_nger

fetch_nger(use_cache=USE_CACHED_NGER)

[NGER-acquire] Acquiring 10 datasets, FY2014-15 → FY2023-24
[NGER-acquire]   FY2014-15 (ID0075) → 424 rows [cached]
[NGER-acquire]   FY2015-16 (ID0076) → 480 rows [cached]
[NGER-acquire]   FY2016-17 (ID0077) → 486 rows [cached]
[NGER-acquire]   FY2017-18 (ID0078) → 522 rows [cached]
[NGER-acquire]   FY2018-19 (ID0079) → 583 rows [cached]
[NGER-acquire]   FY2019-20 (ID0080) → 621 rows [cached]
[NGER-acquire]   FY2020-21 (ID0081) → 655 rows [cached]
[NGER-acquire]   FY2021-22 (ID0082) → 691 rows [cached]
[NGER-acquire]   FY2022-23 (ID0083) → 705 rows [cached]
[NGER-acquire]   FY2023-24 (ID0243) → 775 rows [cached]
[NGER-acquire] Total: 5,942 rows (0 fetched, 10 cached)


### 1.2 CER — Large-scale renewable power stations (4 CSVs scraped from the CER website)

In [ ]:
from pipeline.acquire.cer import fetch_cer

fetch_cer(use_cache=USE_CACHED_CER)

### 1.3 ABS — Data by Region: Population & People (XLSX from the ABS methodology page)

In [ ]:
from pipeline.acquire.abs import fetch_abs

fetch_abs(use_cache=USE_CACHED_ABS)

---

## 2. Data Cleaning

Modules under `pipeline/clean/` normalise column names, coerce types, and write tidy CSVs to `data/cleaned/`.

### 2.1 NGER

In [4]:
from pipeline.clean.nger import clean_nger

df = clean_nger()

df.head()

[NGER-clean] Loading 10 years from data/raw/nger/
[NGER-clean] Concatenated 5,942 rows
[NGER-clean] Row types: {'F': 4862, 'C': 1056, 'FA': 18, '-': 4, '': 2}
[NGER-clean] JV double-count flagged: 30 rows
[NGER-clean] Missing state: 905 (expected ≈ corporate-total rows: 1,056)
[NGER-clean] Saved to data/cleaned/nger.csv (0.72 MB)


,reporting_year,reporting_entity,facility_name,row_type,state,grid_connected,grid,primary_fuel,electricity_production_gj,electricity_production_mwh,scope1_emissions_tco2e,scope2_emissions_tco2e,total_emissions_tco2e,emission_intensity_tco2e_per_mwh,jv_double_counted,important_notes
0,2014-15,ACCIONA ENERGY OCEANIA PTY LTD,Gunning Wind Farm,F,NSW,On,NEM,Wind,567719.0,157700.0,19.0,293.0,312,0.0,False,N/A
1,2014-15,ACCIONA ENERGY OCEANIA PTY LTD,Royalla Solar Farm,F,ACT,On,NEM,Solar,213115.0,59199.0,0.0,15.0,15,0.0,False,N/A
2,2014-15,ACCIONA ENERGY OCEANIA PTY LTD,Waubra Wind Farm,F,VIC,On,NEM,Wind,2461803.0,683834.0,77.0,1144.0,1221,0.0,False,N/A
3,2014-15,ACCIONA ENERGY OCEANIA PTY LTD,Corporate Total,C,N/A,N/A,N/A,N/A,3242637.0,900733.0,96.0,1452.0,1548,NaN,False,N/A
4,2014-15,AGL ENERGY LIMITED,Banimboola Hydro,F,VIC,On,NEM,Hydro,137094.0,38082.0,5.0,17.0,22,0.0,False,N/A


### 2.2 CER

In [ ]:
from pipeline.clean.cer import clean_cer

cer_df = clean_cer()

cer_df.head()

### 2.3 ABS

In [ ]:
from pipeline.clean.abs import clean_abs

abs_df = clean_abs()

abs_df[["code", "label", "year", "geography_level"]].head()

---

## 3. Data Integration

Builds three curated tables in `data/curated/`:

- **`facility`** — one row per unique physical facility, matching NGER `facility_name` to CER `station_name` on (normalised name, state, fuel category).
- **`generation`** — one row per facility-year (NGER `F`/`FA` rows only); FK to `facility`.
- **`abs_population`** — narrow ABS table with a 2-letter `state` derived from the ASGS code, joinable to the others at state level.

In [ ]:
from pipeline.integrate.facility_generation import integrate_facility_generation
from pipeline.integrate.abs_population import integrate_abs_population

facility, generation = integrate_facility_generation()
abs_population = integrate_abs_population()

---

## 4. Data Augmentation — Geocoding (Nominatim)

Adds `lat` / `lon` to facilities flagged `use_for_geocoding == True` (CER `historical_accredited` set, 3,432 rows). Two-tier query: precise facility lookup, then postcode-centroid fallback. Cached per-query under `data/raw/geocode/nominatim_cache.json` — cold run ~3 h, cached run ~1 s.

### 4.0 Run the geocoder (gated)

⚠️ Cold run ≈ **3 hours** (3,432 facilities × 1.1 s rate-limit + Nominatim latency). The shipped cache makes this a no-op (~1 s). Set `RUN_GEOCODE = True` in the cell below to actually fetch — resumable, so cached queries are skipped on subsequent runs.

In [ ]:
from pipeline.augment.geocode import geocode_facilities

# ⚠️  COLD-RUN WARNING: ~3 hours for all 3,432 facilities (Nominatim is
# rate-limited to 1 req/s). With the shipped cache it's a no-op (~1 s).
# Flip to True to actually run the fetch. Resumable — cached queries are
# skipped, so a partial run can be resumed by re-flipping the flag.
RUN_GEOCODE = False

if RUN_GEOCODE:
    cache = geocode_facilities()
else:
    print("Skipped. Set RUN_GEOCODE = True to fetch (cold run ≈ 3 h).")

### 4.1 Coverage breakdown

In [ ]:
import json
from pathlib import Path

cache = json.loads(Path("data/raw/geocode/nominatim_cache.json").read_text())
precise  = sum(1 for v in cache.values() if v and v["source"] == "facility")
fallback = sum(1 for v in cache.values() if v and v["source"] == "postcode")
miss     = sum(1 for v in cache.values() if v is None)

print(f"Cache entries:       {len(cache):,}")
print(f"  Precise hits:      {precise:,}")
print(f"  Fallback hits:     {fallback:,}")
print(f"  Definitive misses: {miss:,}")

### 4.2 Merge geocodes back into `facility.csv`

In [ ]:
from pipeline.augment.merge import merge_geocodes

facility = merge_geocodes()
facility[["facility_id", "facility_name", "state", "lat", "lon", "geocode_source"]].head()

---

## 5. Data Transformation & Storage — DuckDB warehouse

Loads the three curated tables (`facility`, `generation`, `abs_population`) into `data/curated/warehouse.duckdb`. Implementation in `pipeline/load/duckdb_load.py`. The `spatial` extension is loaded so `facility.geom` is stored as `GEOMETRY` (POINT).

In [ ]:
from pipeline.load.duckdb_load import build_warehouse

db_path = build_warehouse()
db_path

### 5.1 Schema inspection

In [ ]:
import duckdb

con = duckdb.connect(str(db_path), read_only=True)
con.execute("LOAD spatial;")

for t in ("facility", "generation", "abs_population"):
    cols = con.execute(f"PRAGMA table_info('{t}')").fetchdf()[["name", "type"]]
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"── {t}  ({n:,} rows)")
    print(cols.to_string(index=False))
    print()